<a href="https://colab.research.google.com/github/svanshika-afk/tinyML-lab-2548544/blob/lab4/2548544_TinyML_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # must be set before importing TF
!pip install -q tensorflow-model-optimization tf_keras
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import numpy as np
import time
print("TensorFlow   :", tf.__version__)
print("TFMOT        :", tfmot.__version__)
print("Keras backend:", tf.keras.backend.backend())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 5.2 MB/s eta 0:00:00
TensorFlow   : 2.20.0
TFMOT        : 0.8.1
Keras backend: tensorflow


In [2]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = \
    tf.keras.datasets.cifar10.load_data()
# ── Subset ──────────────────────────────────────────────────
N_TRAIN, N_TEST = 5000, 1000
x_train_raw = x_train_full[:N_TRAIN]
y_train_raw = y_train_full[:N_TRAIN]
x_test_raw  = x_test_full[:N_TEST]
y_test_raw  = y_test_full[:N_TEST]
# ── Resize 32×32 → 96×96 ────────────────────────────────────
TARGET = (96, 96)
def resize_batch(images):
    return np.array([
        tf.image.resize(img, TARGET).numpy()
        for img in images
    ], dtype=np.float32)
print("Resizing training images …")
x_train_resized = resize_batch(x_train_raw)   # (5000, 96, 96, 3)
print("Resizing test images …")
x_test_resized  = resize_batch(x_test_raw)    # (1000, 96, 96, 3)
# ── MobileNetV2 preprocessing: scale pixels to [–1, 1] ──────
x_train_proc = tf.keras.applications.mobilenet_v2.preprocess_input(
    x_train_resized.copy())
x_test_proc  = tf.keras.applications.mobilenet_v2.preprocess_input(
    x_test_resized.copy())
# ── One-hot labels ───────────────────────────────────────────
NUM_CLASSES = 10
y_train_oh = tf.keras.utils.to_categorical(y_train_raw, NUM_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test_raw,  NUM_CLASSES)
print(f"\nTrain : {x_train_proc.shape} | pixel range [{x_train_proc.min():.2f}, {x_train_proc.max():.2f}]")
print(f"Test  : {x_test_proc.shape}  | labels shape {y_test_oh.shape}")
CLASS_NAMES = ["airplane","automobile","bird","cat","deer",
               "dog","frog","horse","ship","truck"]


170498071/170498071 [==============================] - 154s 1us/step
Resizing training images …
Resizing test images …

Train : (5000, 96, 96, 3) | pixel range [-1.00, 1.00]
Test  : (1000, 96, 96, 3)  | labels shape (1000, 10)


In [3]:
IMG_SHAPE   = (96, 96, 3)
NUM_CLASSES = 10
# ── 1. Create MobileNetV2 base ───────────────────────────────────────────────
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    alpha=0.35,
    include_top=False,
    weights="imagenet"
)
base.trainable = False    # freeze all base layers for Phase 1
# ── 2. Build FLAT functional model using base's own input/output tensors ─────
#    base.input  → the InputLayer tensor already inside MobileNetV2
#    base.output → the feature-map tensor at the end of MobileNetV2
#    Building Model(base.input, ...) inlines every layer — no nesting!
x       = base.output
x       = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
x       = tf.keras.layers.Dropout(0.2, name="dropout")(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax",
                                 name="predictions")(x)
baseline_model = tf.keras.Model(inputs=base.input,
                                outputs=outputs,
                                name="mobilenetv2_baseline")
baseline_model.summary()
# ── Phase 1: Train classification head only ──────────────────────────────────
print("\n── Phase 1: Training head (5 epochs) ──")
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
baseline_model.fit(
    x_train_proc, y_train_oh,
    epochs=5, batch_size=64,
    validation_split=0.1, verbose=1
)
# ── Phase 2: Fine-tune top layers of base ───────────────────────────────────
print("\n── Phase 2: Fine-tuning top layers (10 epochs) ──")
base.trainable   = True
FINE_TUNE_AT     = 100          # freeze the first 100 layers, train the rest
for layer in base.layers[:FINE_TUNE_AT]:
    layer.trainable = False
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),   # lower LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
baseline_model.fit(
    x_train_proc, y_train_oh,
    epochs=10, batch_size=32,
    validation_split=0.1, verbose=1
)
# ── Evaluate ─────────────────────────────────────────────────────────────────
float32_loss, float32_acc = baseline_model.evaluate(
    x_test_proc, y_test_oh, verbose=0)
print(f"\n✅ Float32 Baseline  →  Test Accuracy: {float32_acc*100:.2f}%")
baseline_model.save("baseline_model.h5")
print("Saved: baseline_model.h5")
# Quick sanity check — model should NOT have any nested sub-models
nested = [l for l in baseline_model.layers if isinstance(l, tf.keras.Model)]
print(f"\nNested sub-models found: {len(nested)}  ← must be 0 for QAT to work")


2019640/2019640 [==============================] - 0s 0us/step
Model: "mobilenetv2_baseline"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 48, 48, 16)           432       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 48, 48, 16)           64        ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReL

/usr/local/lib/python3.13/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Saved: baseline_model.h5

Nested sub-models found: 0  ← must be 0 for QAT to work


In [4]:
def get_model_size_kb(path):
    return os.path.getsize(path) / 1024
def evaluate_tflite(model_path, x_data, y_data,
                    num_samples=300, is_int8=False):
    """
    Run TFLite inference and return (accuracy %, avg latency ms).
    is_int8=True  → quantise the float input → int8 before passing in.
    """
    interp = tf.lite.Interpreter(model_path=model_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    if is_int8:
        scale = inp["quantization"][0]
        zp    = inp["quantization"][1]
    correct   = 0
    latencies = []
    n = min(num_samples, len(x_data))
    for i in range(n):
        img = np.expand_dims(x_data[i], axis=0)
        if is_int8:
            img_q = img / scale + zp
            img_q = np.clip(img_q, -128, 127).astype(np.int8)
            interp.set_tensor(inp["index"], img_q)
        else:
            interp.set_tensor(inp["index"], img.astype(np.float32))
        t0 = time.perf_counter()
        interp.invoke()
        latencies.append((time.perf_counter() - t0) * 1000)
        pred  = np.argmax(interp.get_tensor(out["index"])[0])
        truth = np.argmax(y_data[i]) if y_data.ndim > 1 else int(y_data[i])
        if pred == truth:
            correct += 1
    return (correct / n) * 100, float(np.mean(latencies))
print("Helper functions defined ✅")

Helper functions defined ✅


In [5]:

# ── 5-A: Float32 TFLite baseline ────────────────────────────
print("Converting Float32 → TFLite …")
conv_f32 = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
f32_buf  = conv_f32.convert()
f32_path = "baseline_float32.tflite"
with open(f32_path, "wb") as f:
    f.write(f32_buf)
f32_size = get_model_size_kb(f32_path)
print(f"  {f32_path}  →  {f32_size:.1f} KB")
# ── 5-B: Representative dataset ─────────────────────────────
def representative_dataset():
    for i in range(100):
        yield [np.expand_dims(x_train_proc[i], axis=0).astype(np.float32)]
# ── 5-C: Full INT8 PTQ ──────────────────────────────────────
print("\nConverting PTQ Full INT8 …")
conv_ptq = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
conv_ptq.optimizations                = [tf.lite.Optimize.DEFAULT]
conv_ptq.representative_dataset       = representative_dataset
conv_ptq.target_spec.supported_ops    = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_ptq.inference_input_type         = tf.int8
conv_ptq.inference_output_type        = tf.int8
ptq_buf  = conv_ptq.convert()
ptq_path = "ptq_int8.tflite"
with open(ptq_path, "wb") as f:
    f.write(ptq_buf)
ptq_size = get_model_size_kb(ptq_path)
print(f"  {ptq_path}  →  {ptq_size:.1f} KB")
# ── 5-D: Evaluate ───────────────────────────────────────────
print("\nEvaluating Float32 TFLite …")
f32_tflite_acc, f32_latency = evaluate_tflite(
    f32_path, x_test_proc, y_test_oh, num_samples=300, is_int8=False)
print(f"  Accuracy: {f32_tflite_acc:.2f}%  |  Avg latency: {f32_latency:.2f} ms")
print("\nEvaluating PTQ INT8 TFLite …")
ptq_acc, ptq_latency = evaluate_tflite(
    ptq_path, x_test_proc, y_test_oh, num_samples=300, is_int8=True)
print(f"  Accuracy: {ptq_acc:.2f}%  |  Avg latency: {ptq_latency:.2f} ms")


Converting Float32 → TFLite …
  baseline_float32.tflite  →  1597.1 KB

Converting PTQ Full INT8 …


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  ptq_int8.tflite  →  610.2 KB

Evaluating Float32 TFLite …


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  Accuracy: 84.67%  |  Avg latency: 0.64 ms

Evaluating PTQ INT8 TFLite …
  Accuracy: 51.33%  |  Avg latency: 1.10 ms


In [6]:
# ── 6-A: Apply QAT ──────────────────────────────────────────
print("Applying QAT to baseline model …")
qat_model = tfmot.quantization.keras.quantize_model(baseline_model)
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   # low LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
qat_model.summary()
# ── 6-B: Fine-tune with fake-quant nodes ────────────────────
print("\n── QAT Fine-tuning (5 epochs) ──")
qat_model.fit(
    x_train_proc, y_train_oh,
    epochs=5, batch_size=32,
    validation_split=0.1, verbose=1
)
# ── 6-C: Convert to Full INT8 TFLite ────────────────────────
print("\nConverting QAT → Full INT8 TFLite …")
conv_qat = tf.lite.TFLiteConverter.from_keras_model(qat_model)
conv_qat.optimizations             = [tf.lite.Optimize.DEFAULT]
conv_qat.representative_dataset    = representative_dataset  # needed for INT8 I/O
conv_qat.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_qat.inference_input_type      = tf.int8
conv_qat.inference_output_type     = tf.int8
qat_buf  = conv_qat.convert()
qat_path = "qat_int8.tflite"
with open(qat_path, "wb") as f:
    f.write(qat_buf)
qat_size = get_model_size_kb(qat_path)
print(f"  {qat_path}  →  {qat_size:.1f} KB")
# ── 6-D: Evaluate ───────────────────────────────────────────
print("\nEvaluating QAT INT8 TFLite …")
qat_acc, qat_latency = evaluate_tflite(
    qat_path, x_test_proc, y_test_oh, num_samples=300, is_int8=True)
print(f"  Accuracy: {qat_acc:.2f}%  |  Avg latency: {qat_latency:.2f} ms")
# %% ── Cell 7: Comparison Table ─────────────────────────────
SEP = "=" * 68
print(f"\n{SEP}")
print(f"{'Model':<26} {'Size (KB)':>10} {'Accuracy (%)':>14} {'Latency (ms)':>14}")
print(SEP)
print(f"{'Float32 Baseline':<26} {f32_size:>10.1f} {f32_tflite_acc:>14.2f} {f32_latency:>14.2f}")
print(f"{'PTQ Full INT8':<26} {ptq_size:>10.1f} {ptq_acc:>14.2f} {ptq_latency:>14.2f}")
print(f"{'QAT Full INT8':<26} {qat_size:>10.1f} {qat_acc:>14.2f} {qat_latency:>14.2f}")
print(SEP)
# Download .tflite files (Colab only)
try:
    from google.colab import files
    for p in [f32_path, ptq_path, qat_path]:
        files.download(p)
    print("\n✅ All three .tflite files downloaded.")
except ImportError:
    print("\n(Not in Colab – files saved to local directory.)")


Applying QAT to baseline model …
Model: "mobilenetv2_baseline"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 quantize_layer (QuantizeLa  (None, 96, 96, 3)            3         ['input_1[0][0]']             
 yer)                                                                                             
                                                                                                  
 quant_Conv1 (QuantizeWrapp  (None, 48, 48, 16)           465       ['quantize_layer[0][0]']      
 erV2)                                                                                            
                                              

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  qat_int8.tflite  →  617.1 KB

Evaluating QAT INT8 TFLite …


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  Accuracy: 83.00%  |  Avg latency: 1.07 ms

Model                       Size (KB)   Accuracy (%)   Latency (ms)
Float32 Baseline               1597.1          84.67           0.64
PTQ Full INT8                   610.2          51.33           1.10
QAT Full INT8                   617.1          83.00           1.07


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All three .tflite files downloaded.
